In [ ]:
import sys
import importlib
from pathlib import Path
from datetime import datetime
import json
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

target_path = "/content/drive/MyDrive/Thesis/models"

if target_path not in sys.path:
    sys.path.insert(0, target_path)

importlib.invalidate_caches()

from utils import DATA_DIR, EKMAN_LABELS, set_all_seeds
from evaluate import compute_metrics

from train_phase3 import (
    GoEmotionsDataset,
    hf_compute_metrics,
    load_data,
    MODEL_NAME,
    MAX_LENGTH,
    BATCH_SIZE,
    N_EPOCHS,
    LEARNING_RATE,
    WEIGHT_DECAY,
    WARMUP_STEPS,
    NUM_LABELS,
)

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SEEDS = [42, 0, 1]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def make_json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): make_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_safe(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    return value


def get_class_counts(labels, num_labels=NUM_LABELS):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    counts = np.bincount(
        labels,
        minlength=num_labels,
    )

    if np.any(counts == 0):
        missing = np.where(counts == 0)[0].tolist()

        raise ValueError(
            f"Missing classes in training labels: {missing}"
        )

    return counts


def normalize_by_expected_example_weight(
    weights,
    counts,
):
    weights = np.asarray(
        weights,
        dtype=np.float64,
    )

    counts = np.asarray(
        counts,
        dtype=np.float64,
    )

    expected_weight = (
        np.sum(weights * counts)
        / np.sum(counts)
    )

    return weights / expected_weight


def save_confusion_matrix_to_folder(
    y_true,
    y_pred,
    labels,
    output_path,
    title,
):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(labels))),
    )

    figure, axis = plt.subplots(
        figsize=(9, 8)
    )

    image = axis.imshow(
        matrix,
        interpolation="nearest",
        cmap="Blues",
    )

    figure.colorbar(
        image,
        ax=axis,
    )

    axis.set(
        xticks=np.arange(len(labels)),
        yticks=np.arange(len(labels)),
        xticklabels=labels,
        yticklabels=labels,
        xlabel="Predicted label",
        ylabel="True label",
        title=title,
    )

    plt.setp(
        axis.get_xticklabels(),
        rotation=45,
        ha="right",
        rotation_mode="anchor",
    )

    threshold = matrix.max() / 2 if matrix.size else 0

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            axis.text(
                column_index,
                row_index,
                format(matrix[row_index, column_index], "d"),
                ha="center",
                va="center",
                color=(
                    "white"
                    if matrix[row_index, column_index] > threshold
                    else "black"
                ),
            )

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)

    return matrix

In [ ]:
def balanced_inverse_frequency_alpha(
    labels,
    num_labels=NUM_LABELS,
):
    counts = get_class_counts(
        labels,
        num_labels=num_labels,
    )

    n_examples = counts.sum()
    n_classes = num_labels

    alpha = n_examples / (
        n_classes * counts
    )

    alpha = normalize_by_expected_example_weight(
        alpha,
        counts,
    )

    return alpha, counts


def sqrt_inverse_frequency_alpha(
    labels,
    num_labels=NUM_LABELS,
):
    alpha, counts = balanced_inverse_frequency_alpha(
        labels,
        num_labels=num_labels,
    )

    alpha = np.sqrt(
        alpha
    )

    alpha = normalize_by_expected_example_weight(
        alpha,
        counts,
    )

    return alpha, counts


def capped_inverse_frequency_alpha(
    labels,
    cap=3.0,
    num_labels=NUM_LABELS,
):
    alpha, counts = balanced_inverse_frequency_alpha(
        labels,
        num_labels=num_labels,
    )

    alpha = np.minimum(
        alpha,
        cap,
    )

    alpha = normalize_by_expected_example_weight(
        alpha,
        counts,
    )

    return alpha, counts


def effective_number_alpha(
    labels,
    beta=0.999,
    num_labels=NUM_LABELS,
):
    counts = get_class_counts(
        labels,
        num_labels=num_labels,
    ).astype(np.float64)

    effective_num = (
        1.0 - np.power(beta, counts)
    ) / (1.0 - beta)

    alpha = 1.0 / effective_num

    alpha = normalize_by_expected_example_weight(
        alpha,
        counts,
    )

    return alpha, counts


def compute_alpha_scheme(
    labels,
    scheme_name,
    **kwargs,
):
    if scheme_name == "sqrt_inverse_frequency_alpha":
        return sqrt_inverse_frequency_alpha(
            labels,
        )

    if scheme_name == "capped_inverse_frequency_alpha":
        return capped_inverse_frequency_alpha(
            labels,
            cap=kwargs.get("cap", 3.0),
        )

    if scheme_name == "effective_number_alpha":
        return effective_number_alpha(
            labels,
            beta=kwargs.get("beta", 0.999),
        )

    raise ValueError(
        f"Unknown alpha scheme: {scheme_name}"
    )

In [ ]:
class AlphaBalancedFocalLoss(nn.Module):

    def __init__(
        self,
        gamma=2.0,
        alpha_vector=None,
        reduction="mean",
    ):
        super().__init__()

        if gamma < 0:
            raise ValueError(
                "gamma must be greater than or equal to zero."
            )

        if reduction not in {
            "none",
            "mean",
            "sum",
        }:
            raise ValueError(
                "Invalid reduction."
            )

        if alpha_vector is None:
            alpha_vector = torch.ones(
                NUM_LABELS,
                dtype=torch.float32,
            )

        if not isinstance(
            alpha_vector,
            torch.Tensor,
        ):
            alpha_vector = torch.tensor(
                alpha_vector,
                dtype=torch.float32,
            )

        if alpha_vector.ndim != 1:
            raise ValueError(
                "alpha_vector must be one-dimensional."
            )

        if len(alpha_vector) != NUM_LABELS:
            raise ValueError(
                f"Expected alpha_vector of length {NUM_LABELS}, "
                f"but got {len(alpha_vector)}."
            )

        if torch.any(alpha_vector <= 0):
            raise ValueError(
                "All alpha values must be positive."
            )

        self.gamma = float(gamma)
        self.reduction = reduction

        self.register_buffer(
            "alpha_vector",
            alpha_vector.to(dtype=torch.float32),
        )

    def forward(
        self,
        logits,
        targets,
    ):
        targets = targets.long()

        cross_entropy = F.cross_entropy(
            logits,
            targets,
            reduction="none",
        )

        pt = torch.exp(
            -cross_entropy
        )

        focal_factor = (
            1.0 - pt
        ).pow(self.gamma)

        alpha_t = self.alpha_vector.to(
            logits.device
        )[targets]

        loss = (
            alpha_t
            * focal_factor
            * cross_entropy
        )

        if self.reduction == "none":
            return loss

        if self.reduction == "sum":
            return loss.sum()

        return loss.mean()

In [ ]:
class AlphaFocalTrainer(Trainer):
    def __init__(
        self,
        *args,
        gamma=2.0,
        alpha_vector=None,
        **kwargs,
    ):
        super().__init__(
            *args,
            **kwargs,
        )

        self.loss_fn = AlphaBalancedFocalLoss(
            gamma=gamma,
            alpha_vector=alpha_vector,
            reduction="mean",
        )

        self.model_accepts_loss_kwargs = False

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs,
    ):
        model_inputs = dict(inputs)

        labels = model_inputs.pop(
            "labels"
        ).long()

        outputs = model(
            **model_inputs
        )

        logits = outputs.logits

        loss = self.loss_fn(
            logits,
            labels,
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

In [ ]:
def run_alpha_focal_experiment(
    config,
    seed=42,
    output_root=OUTPUT_ROOT,
):
    set_all_seeds(seed)

    experiment_name = config["experiment_name"]
    alpha_scheme = config["alpha_scheme"]
    gamma = config["gamma"]

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    run_name = (
        f"phase4_alpha_focal_{experiment_name}_"
        f"seed{seed}_{timestamp}"
    )

    out_dir = Path(output_root) / run_name

    if out_dir.exists():
        raise FileExistsError(
            f"Output folder already exists:\n{out_dir}"
        )

    out_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    print("=" * 80)
    print("PHASE 4 ALPHA-BALANCED FOCAL CONTROL")
    print("=" * 80)
    print(f"Experiment: {experiment_name}")
    print(f"Alpha:      {alpha_scheme}")
    print(f"Gamma:      {gamma}")
    print(f"Seed:       {seed}")
    print(f"Output:     {out_dir}")
    print("=" * 80)

    train_df, val_df, test_df = load_data()

    alpha, counts = compute_alpha_scheme(
        train_df["ekman_id"].values,
        scheme_name=alpha_scheme,
        **config.get("parameters", {}),
    )

    alpha_tensor = torch.tensor(
        alpha,
        dtype=torch.float32,
    )

    alpha_report = {
        "experiment_name": experiment_name,
        "alpha_scheme": alpha_scheme,
        "gamma": gamma,
        "parameters": config.get("parameters", {}),
        "class_counts": {
            EKMAN_LABELS[index]: int(count)
            for index, count in enumerate(counts)
        },
        "alpha_values": {
            EKMAN_LABELS[index]: float(value)
            for index, value in enumerate(alpha)
        },
        "max_min_alpha_ratio": float(
            np.max(alpha) / np.min(alpha)
        ),
    }

    print("\nAlpha values:")
    for index, label in enumerate(EKMAN_LABELS):
        print(
            f"{label:<10} count={int(counts[index]):<6} "
            f"alpha={alpha[index]:.6f}"
        )

    print(
        f"\nMax/min alpha ratio: "
        f"{alpha_report['max_min_alpha_ratio']:.3f}"
    )

    with open(
        out_dir / "alpha_report.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            make_json_safe(alpha_report),
            file,
            indent=2,
            ensure_ascii=False,
        )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label={
            index: label
            for index, label in enumerate(EKMAN_LABELS)
        },
        label2id={
            label: index
            for index, label in enumerate(EKMAN_LABELS)
        },
    )

    train_ds = GoEmotionsDataset(
        train_df["text_clean"],
        train_df["ekman_id"],
        tokenizer,
    )

    val_ds = GoEmotionsDataset(
        val_df["text_clean"],
        val_df["ekman_id"],
        tokenizer,
    )

    test_ds = GoEmotionsDataset(
        test_df["text_clean"],
        test_df["ekman_id"],
        tokenizer,
    )

    training_args = TrainingArguments(
        output_dir=str(out_dir / "checkpoints"),
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=N_EPOCHS,
        warmup_steps=WARMUP_STEPS,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        logging_steps=100,
        seed=seed,
        data_seed=seed,
        report_to="none",
    )

    trainer = AlphaFocalTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(
            tokenizer=tokenizer
        ),
        compute_metrics=hf_compute_metrics,
        gamma=gamma,
        alpha_vector=alpha_tensor,
    )

    print("\nStarting training...")

    training_start = time.time()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    trainer.train()

    training_time = time.time() - training_start

    peak_gpu_memory_gb = None

    if torch.cuda.is_available():
        peak_gpu_memory_gb = (
            torch.cuda.max_memory_allocated() / 1e9
        )

    best_model_dir = out_dir / "best_model"

    trainer.save_model(
        str(best_model_dir)
    )

    tokenizer.save_pretrained(
        str(best_model_dir)
    )

    def evaluate_and_save(
        dataset,
        dataframe,
        split_name,
    ):
        prediction_output = trainer.predict(
            dataset,
            metric_key_prefix=split_name,
        )

        logits = prediction_output.predictions

        if isinstance(logits, tuple):
            logits = logits[0]

        y_true = prediction_output.label_ids.astype(int)
        y_pred = np.argmax(
            logits,
            axis=-1,
        ).astype(int)

        stable_logits = (
            logits
            - np.max(
                logits,
                axis=1,
                keepdims=True,
            )
        )

        exponentials = np.exp(stable_logits)

        probabilities = (
            exponentials
            / np.sum(
                exponentials,
                axis=1,
                keepdims=True,
            )
        )

        split_metrics = compute_metrics(
            y_true,
            y_pred,
            EKMAN_LABELS,
        )

        prediction_columns = {
            "text_clean": dataframe["text_clean"]
                .astype(str)
                .tolist(),
            "true_id": y_true,
            "true_label": [
                EKMAN_LABELS[value]
                for value in y_true
            ],
            "predicted_id": y_pred,
            "predicted_label": [
                EKMAN_LABELS[value]
                for value in y_pred
            ],
            "correct": y_true == y_pred,
        }

        if "id" in dataframe.columns:
            prediction_columns = {
                "id": dataframe["id"].tolist(),
                **prediction_columns,
            }

        predictions_dataframe = pd.DataFrame(
            prediction_columns
        )

        for class_index, class_name in enumerate(EKMAN_LABELS):
            safe_class_name = (
                str(class_name)
                .lower()
                .replace(" ", "_")
            )

            predictions_dataframe[
                f"probability_{safe_class_name}"
            ] = probabilities[:, class_index]

        predictions_dataframe.to_csv(
            out_dir / f"{split_name}_predictions.csv",
            index=False,
        )

        matrix = save_confusion_matrix_to_folder(
            y_true=y_true,
            y_pred=y_pred,
            labels=EKMAN_LABELS,
            output_path=(
                out_dir
                / f"{split_name}_confusion_matrix.png"
            ),
            title=f"{split_name.capitalize()} confusion matrix",
        )

        np.savetxt(
            out_dir / f"{split_name}_confusion_matrix.csv",
            matrix,
            delimiter=",",
            fmt="%d",
        )

        with open(
            out_dir / f"{split_name}_metrics.json",
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                make_json_safe(split_metrics),
                file,
                indent=2,
                ensure_ascii=False,
            )

        return split_metrics

    validation_metrics = evaluate_and_save(
        val_ds,
        val_df,
        "validation",
    )

    test_metrics = evaluate_and_save(
        test_ds,
        test_df,
        "test",
    )

    complete_results = {
        "run_name": run_name,
        "experiment_name": experiment_name,
        "treatment": "alpha_balanced_focal_loss",
        "seed": seed,
        "model_name": MODEL_NAME,
        "alpha_report": alpha_report,
        "gamma": gamma,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_validation_macro_f1_during_training": trainer.state.best_metric,
        "training_time_seconds": training_time,
        "peak_gpu_memory_gb": peak_gpu_memory_gb,
        "validation": validation_metrics,
        "test": test_metrics,
    }

    with open(
        out_dir / "complete_results.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            make_json_safe(complete_results),
            file,
            indent=2,
            ensure_ascii=False,
        )

    with open(
        out_dir / "training_log_history.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            make_json_safe(trainer.state.log_history),
            file,
            indent=2,
            ensure_ascii=False,
        )

    print("\nValidation macro-F1:")
    print(validation_metrics["macro_f1"])

    print("\nTest macro-F1:")
    print(test_metrics["macro_f1"])

    print("\nResults saved in:")
    print(out_dir)

    return complete_results

In [ ]:
ALPHA_FOCAL_CONFIGS = [
    {
        "experiment_name": "gamma1_alpha_sqrt_inverse_frequency",
        "gamma": 1.0,
        "alpha_scheme": "sqrt_inverse_frequency_alpha",
        "parameters": {},
    },
    {
        "experiment_name": "gamma2_alpha_sqrt_inverse_frequency",
        "gamma": 2.0,
        "alpha_scheme": "sqrt_inverse_frequency_alpha",
        "parameters": {},
    },
    {
        "experiment_name": "gamma1_alpha_capped_cap3",
        "gamma": 1.0,
        "alpha_scheme": "capped_inverse_frequency_alpha",
        "parameters": {
            "cap": 3.0,
        },
    },
    {
        "experiment_name": "gamma2_alpha_capped_cap3",
        "gamma": 2.0,
        "alpha_scheme": "capped_inverse_frequency_alpha",
        "parameters": {
            "cap": 3.0,
        },
    },
    {
        "experiment_name": "gamma1_alpha_effective_beta0999",
        "gamma": 1.0,
        "alpha_scheme": "effective_number_alpha",
        "parameters": {
            "beta": 0.999,
        },
    },
    {
        "experiment_name": "gamma2_alpha_effective_beta0999",
        "gamma": 2.0,
        "alpha_scheme": "effective_number_alpha",
        "parameters": {
            "beta": 0.999,
        },
    },
]

In [ ]:
def run_one_alpha_focal_config_for_all_seeds(
    experiment_name,
    seeds=SEEDS,
):
    matching_configs = [
        config
        for config in ALPHA_FOCAL_CONFIGS
        if config["experiment_name"] == experiment_name
    ]

    if len(matching_configs) != 1:
        raise ValueError(
            f"Could not find exactly one config for: {experiment_name}"
        )

    config = matching_configs[0]

    results = []

    for seed in seeds:
        result = run_alpha_focal_experiment(
            config=config,
            seed=seed,
            output_root=OUTPUT_ROOT,
        )

        results.append(result)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return results

In [ ]:
af_g1_sqrt_results = run_one_alpha_focal_config_for_all_seeds(
    "gamma1_alpha_sqrt_inverse_frequency"
)

PHASE 4 ALPHA-BALANCED FOCAL CONTROL
Experiment: gamma1_alpha_sqrt_inverse_frequency
Alpha:      sqrt_inverse_frequency_alpha
Gamma:      1.0
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs/phase4_alpha_focal_gamma1_alpha_sqrt_inverse_frequency_seed42_20260702_084815

Alpha values:
anger      count=3878   alpha=1.325460
disgust    count=497    alpha=3.702477
fear       count=515    alpha=3.637198
joy        count=12920  alpha=0.726172
sadness    count=2121   alpha=1.792257
surprise   count=3553   alpha=1.384755
neutral    count=12818  alpha=0.729055

Max/min alpha ratio: 5.099


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6224724420419321

Test macro-F1:
0.6222157331395682

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs/phase4_alpha_focal_gamma1_alpha_sqrt_inverse_frequency_seed1_20260702_090211


In [ ]:
af_g2_sqrt_results = run_one_alpha_focal_config_for_all_seeds(
    "gamma2_alpha_sqrt_inverse_frequency"
)

PHASE 4 ALPHA-BALANCED FOCAL CONTROL
Experiment: gamma2_alpha_sqrt_inverse_frequency
Alpha:      sqrt_inverse_frequency_alpha
Gamma:      2.0
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs/phase4_alpha_focal_gamma2_alpha_sqrt_inverse_frequency_seed42_20260702_090920

Alpha values:
anger      count=3878   alpha=1.325460
disgust    count=497    alpha=3.702477
fear       count=515    alpha=3.637198
joy        count=12920  alpha=0.726172
sadness    count=2121   alpha=1.792257
surprise   count=3553   alpha=1.384755
neutral    count=12818  alpha=0.729055

Max/min alpha ratio: 5.099


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.584118,0.533437,0.672091,0.600090,0.679380
2,0.479208,0.516256,0.682648,0.620571,0.686836
3,0.346434,0.531397,0.677590,0.608307,0.680863


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6199426079555673

Test macro-F1:
0.6245857978173689

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs/phase4_alpha_focal_gamma2_alpha_sqrt_inverse_frequency_seed1_20260702_092312


In [ ]:
af_g1_cap3_results = run_one_alpha_focal_config_for_all_seeds(
    "gamma1_alpha_capped_cap3"
)

PHASE 4 ALPHA-BALANCED FOCAL CONTROL
Experiment: gamma1_alpha_capped_cap3
Alpha:      capped_inverse_frequency_alpha
Gamma:      1.0
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs/phase4_alpha_focal_gamma1_alpha_capped_cap3_seed42_20260702_093019

Alpha values:
anger      count=3878   alpha=1.675972
disgust    count=497    alpha=3.759787
fear       count=515    alpha=3.759787
joy        count=12920  alpha=0.503051
sadness    count=2121   alpha=3.064318
surprise   count=3553   alpha=1.829276
neutral    count=12818  alpha=0.507054

Max/min alpha ratio: 7.474


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.770893,0.707411,0.638443,0.583487,0.649527
2,0.625196,0.688458,0.661095,0.605868,0.668311
3,0.480379,0.712044,0.661755,0.606086,0.667130


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6084608382759408

Test macro-F1:
0.6048353446192895

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs/phase4_alpha_focal_gamma1_alpha_capped_cap3_seed1_20260702_094421


In [ ]:
af_g2_cap3_results = run_one_alpha_focal_config_for_all_seeds(
    "gamma2_alpha_capped_cap3"
)

PHASE 4 ALPHA-BALANCED FOCAL CONTROL
Experiment: gamma2_alpha_capped_cap3
Alpha:      capped_inverse_frequency_alpha
Gamma:      2.0
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs/phase4_alpha_focal_gamma2_alpha_capped_cap3_seed42_20260702_095119

Alpha values:
anger      count=3878   alpha=1.675972
disgust    count=497    alpha=3.759787
fear       count=515    alpha=3.759787
joy        count=12920  alpha=0.503051
sadness    count=2121   alpha=3.064318
surprise   count=3553   alpha=1.829276
neutral    count=12818  alpha=0.507054

Max/min alpha ratio: 7.474


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.611603,0.559560,0.636684,0.578503,0.647480
2,0.485279,0.541982,0.660216,0.610312,0.667066
3,0.354377,0.565138,0.658236,0.603252,0.663301


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6115414887894993

Test macro-F1:
0.6057115283747437

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs/phase4_alpha_focal_gamma2_alpha_capped_cap3_seed1_20260702_100500


In [ ]:
af_g1_eff_results = run_one_alpha_focal_config_for_all_seeds(
    "gamma1_alpha_effective_beta0999"
)

PHASE 4 ALPHA-BALANCED FOCAL CONTROL
Experiment: gamma1_alpha_effective_beta0999
Alpha:      effective_number_alpha
Gamma:      1.0
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs/phase4_alpha_focal_gamma1_alpha_effective_beta0999_seed42_20260702_101218

Alpha values:
anger      count=3878   alpha=0.967503
disgust    count=497    alpha=2.418392
fear       count=515    alpha=2.353195
joy        count=12920  alpha=0.947524
sadness    count=2121   alpha=1.076466
surprise   count=3553   alpha=0.975407
neutral    count=12818  alpha=0.947525

Max/min alpha ratio: 2.552


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.654592,0.610043,0.687046,0.600513,0.686542
2,0.558718,0.595135,0.699142,0.628827,0.698823
3,0.442777,0.606640,0.701561,0.628724,0.701782


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6289910264906233

Test macro-F1:
0.6337564898951233

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs/phase4_alpha_focal_gamma1_alpha_effective_beta0999_seed1_20260702_102616


In [ ]:
af_g2_eff_results = run_one_alpha_focal_config_for_all_seeds(
    "gamma2_alpha_effective_beta0999"
)

PHASE 4 ALPHA-BALANCED FOCAL CONTROL
Experiment: gamma2_alpha_effective_beta0999
Alpha:      effective_number_alpha
Gamma:      2.0
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs/phase4_alpha_focal_gamma2_alpha_effective_beta0999_seed42_20260702_103333

Alpha values:
anger      count=3878   alpha=0.967503
disgust    count=497    alpha=2.418392
fear       count=515    alpha=2.353195
joy        count=12920  alpha=0.947524
sadness    count=2121   alpha=1.076466
surprise   count=3553   alpha=0.975407
neutral    count=12818  alpha=0.947525

Max/min alpha ratio: 2.552


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.503600,0.462823,0.683748,0.594177,0.682541
2,0.421005,0.448989,0.697383,0.624612,0.697313
3,0.316035,0.458412,0.698483,0.628644,0.698667


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6249050523490857

Test macro-F1:
0.6327035368588086

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_alpha_focal_control_runs/phase4_alpha_focal_gamma2_alpha_effective_beta0999_seed1_20260702_104729


In [ ]:
def collect_alpha_focal_results(
    output_root=OUTPUT_ROOT,
):
    rows = []

    for result_file in Path(output_root).glob(
        "phase4_alpha_focal_*/complete_results.json"
    ):
        with open(
            result_file,
            "r",
            encoding="utf-8",
        ) as file:
            result = json.load(file)

        alpha_report = result["alpha_report"]

        rows.append({
            "run_name": result["run_name"],
            "experiment_name": result["experiment_name"],
            "seed": result["seed"],
            "gamma": result["gamma"],
            "max_min_alpha_ratio": alpha_report["max_min_alpha_ratio"],
            "best_validation_macro_f1": result[
                "best_validation_macro_f1_during_training"
            ],
            "validation_macro_f1": result["validation"]["macro_f1"],
            "validation_accuracy": result["validation"]["accuracy"],
            "test_macro_f1": result["test"]["macro_f1"],
            "test_accuracy": result["test"]["accuracy"],
            "path": str(result_file.parent),
        })

    summary = pd.DataFrame(rows)

    if len(summary) == 0:
        raise FileNotFoundError(
            "No alpha-focal results found."
        )

    summary = summary.sort_values(
        [
            "experiment_name",
            "seed",
        ]
    ).reset_index(drop=True)

    summary.to_csv(
        Path(output_root)
        / "alpha_focal_controls_by_seed.csv",
        index=False,
    )

    return summary


alpha_focal_by_seed = collect_alpha_focal_results(
    OUTPUT_ROOT
)

alpha_focal_by_seed

,run_name,experiment_name,seed,gamma,max_min_alpha_ratio,best_validation_macro_f1,validation_macro_f1,validation_accuracy,test_macro_f1,test_accuracy,path
0,phase4_alpha_focal_gamma1_alpha_capped_cap3_se...,gamma1_alpha_capped_cap3,0,1.0,7.473968,0.610582,0.610582,0.668133,0.603226,0.653595,/content/drive/MyDrive/Thesis/models/phase4_al...
1,phase4_alpha_focal_gamma1_alpha_capped_cap3_se...,gamma1_alpha_capped_cap3,1,1.0,7.473968,0.608461,0.608461,0.664614,0.604835,0.650109,/content/drive/MyDrive/Thesis/models/phase4_al...
2,phase4_alpha_focal_gamma1_alpha_capped_cap3_se...,gamma1_alpha_capped_cap3,42,1.0,7.473968,0.606086,0.606086,0.661755,0.603181,0.647495,/content/drive/MyDrive/Thesis/models/phase4_al...
3,phase4_alpha_focal_gamma1_alpha_effective_beta...,gamma1_alpha_effective_beta0999,0,1.0,2.552328,0.630821,0.630821,0.697163,0.625763,0.686492,/content/drive/MyDrive/Thesis/models/phase4_al...
4,phase4_alpha_focal_gamma1_alpha_effective_beta...,gamma1_alpha_effective_beta0999,1,1.0,2.552328,0.628991,0.628991,0.700242,0.633756,0.694118,/content/drive/MyDrive/Thesis/models/phase4_al...
5,phase4_alpha_focal_gamma1_alpha_effective_beta...,gamma1_alpha_effective_beta0999,42,1.0,2.552328,0.628827,0.628827,0.699142,0.641157,0.697603,/content/drive/MyDrive/Thesis/models/phase4_al...
6,phase4_alpha_focal_gamma1_alpha_sqrt_inverse_f...,gamma1_alpha_sqrt_inverse_frequency,0,1.0,5.098625,0.619576,0.619576,0.689246,0.618981,0.679085,/content/drive/MyDrive/Thesis/models/phase4_al...
7,phase4_alpha_focal_gamma1_alpha_sqrt_inverse_f...,gamma1_alpha_sqrt_inverse_frequency,1,1.0,5.098625,0.622472,0.622472,0.685727,0.622216,0.674074,/content/drive/MyDrive/Thesis/models/phase4_al...
8,phase4_alpha_focal_gamma1_alpha_sqrt_inverse_f...,gamma1_alpha_sqrt_inverse_frequency,42,1.0,5.098625,0.622548,0.622548,0.685507,0.628169,0.679739,/content/drive/MyDrive/Thesis/models/phase4_al...
9,phase4_alpha_focal_gamma2_alpha_capped_cap3_se...,gamma2_alpha_capped_cap3,0,2.0,7.473968,0.608248,0.608248,0.663514,0.604951,0.645752,/content/drive/MyDrive/Thesis/models/phase4_al...


In [ ]:
alpha_focal_mean_std = (
    alpha_focal_by_seed
    .groupby("experiment_name")
    .agg(
        seeds=("seed", "count"),
        gamma=("gamma", "first"),
        max_min_alpha_ratio_mean=("max_min_alpha_ratio", "mean"),
        validation_macro_f1_mean=("validation_macro_f1", "mean"),
        validation_macro_f1_std=("validation_macro_f1", "std"),
        test_macro_f1_mean=("test_macro_f1", "mean"),
        test_macro_f1_std=("test_macro_f1", "std"),
        test_accuracy_mean=("test_accuracy", "mean"),
        test_accuracy_std=("test_accuracy", "std"),
    )
    .reset_index()
    .sort_values(
        "validation_macro_f1_mean",
        ascending=False,
    )
)

alpha_focal_mean_std.to_csv(
    OUTPUT_ROOT / "alpha_focal_controls_mean_std.csv",
    index=False,
)

alpha_focal_mean_std

,experiment_name,seeds,gamma,max_min_alpha_ratio_mean,validation_macro_f1_mean,validation_macro_f1_std,test_macro_f1_mean,test_macro_f1_std,test_accuracy_mean,test_accuracy_std
1,gamma1_alpha_effective_beta0999,3,1.0,2.552328,0.629546,0.001107,0.633559,0.007699,0.692738,0.005683
4,gamma2_alpha_effective_beta0999,3,2.0,2.552328,0.627119,0.001962,0.628899,0.008440,0.689615,0.002801
2,gamma1_alpha_sqrt_inverse_frequency,3,1.0,5.098625,0.621532,0.001694,0.623122,0.004661,0.677633,0.003099
5,gamma2_alpha_sqrt_inverse_frequency,3,2.0,5.098625,0.620324,0.000335,0.624660,0.006744,0.676325,0.002724
3,gamma2_alpha_capped_cap3,3,2.0,7.473968,0.610034,0.001664,0.607101,0.003088,0.648947,0.002801
0,gamma1_alpha_capped_cap3,3,1.0,7.473968,0.608376,0.002249,0.603747,0.000942,0.650399,0.003060
